## 1. Connect Google Drive

Run the next cell to mount the Drive account containing the LEVIR-CD dataset.

In [ ]:
# Try mounting again. Make sure to complete the authentication pop-up windows!
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully!")
except Exception as e:
    print(f"Mount failed with error: {e}")
    print("Troubleshooting:\n1. Ensure you allow pop-ups and accept the Google authentication prompt.\n2. If using Brave/Safari, disable shields or allow third-party cookies.\n3. You can also click the folder icon on the left panel of Colab and click 'Mount Drive'.")

Mounted at /content/drive
Drive mounted successfully!


## 2. Set the LEVIR-CD dataset path

Set `drive_dataset_path` to the directory that contains the `train`, `val` and `test` folders.

In [ ]:
import os
from glob import glob

# Adjust this path to match your dataset location in your Drive
drive_dataset_path = '/content/drive/MyDrive/LEVIR-CD/LEVIR CD'

# Verify path existence
if not os.path.exists(drive_dataset_path):
    print(f"Warning: The path '{drive_dataset_path}' was not found. Please verify the spelling and folder location inside your Google Drive.")
else:
    print(f"Dataset folder successfully located at: {drive_dataset_path}")

Dataset folder successfully located at: /content/drive/MyDrive/LEVIR-CD/LEVIR CD


## 3. Read the image file paths

The next cell lists the image files and reports the number of samples in each split.

In [ ]:
train_path = f'{drive_dataset_path}/train'
val_path = f'{drive_dataset_path}/val'
test_path = f'{drive_dataset_path}/test'

train_A = sorted(glob(f'{train_path}/A/*.png'))
train_B = sorted(glob(f'{train_path}/B/*.png'))
train_label = sorted(glob(f'{train_path}/label/*.png'))

val_A = sorted(glob(f'{val_path}/A/*.png'))
val_B = sorted(glob(f'{val_path}/B/*.png'))
val_label = sorted(glob(f'{val_path}/label/*.png'))

test_A = sorted(glob(f'{test_path}/A/*.png'))
test_B = sorted(glob(f'{test_path}/B/*.png'))
test_label = sorted(glob(f'{test_path}/label/*.png'))

IMG_SIZE = 128
print(f"Found: Train images: {len(train_A)}, Validation images: {len(val_A)}, Test images: {len(test_A)}")

Found: Train images: 445, Validation images: 64, Test images: 128


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import cv2
from glob import glob
import matplotlib.pyplot as plt

In [ ]:
# Point directly to your successfully mounted Google Drive location
train_path = '/content/drive/MyDrive/LEVIR-CD/LEVIR CD/train'
val_path = '/content/drive/MyDrive/LEVIR-CD/LEVIR CD/val'
test_path = '/content/drive/MyDrive/LEVIR-CD/LEVIR CD/test'

# Glob and sort images
train_A = sorted(glob(f'{train_path}/A/*.png'))
train_B = sorted(glob(f'{train_path}/B/*.png'))
train_label = sorted(glob(f'{train_path}/label/*.png'))

val_A = sorted(glob(f'{val_path}/A/*.png'))
val_B = sorted(glob(f'{val_path}/B/*.png'))
val_label = sorted(glob(f'{val_path}/label/*.png'))

test_A = sorted(glob(f'{test_path}/A/*.png'))
test_B = sorted(glob(f'{test_path}/B/*.png'))
test_label = sorted(glob(f'{test_path}/label/*.png'))

IMG_SIZE = 128
print(f"Found: Train images: {len(train_A)}, Validation images: {len(val_A)}, Test images: {len(test_A)}")

Found: Train images: 445, Validation images: 64, Test images: 128


In [ ]:
def load_data(A_paths, B_paths, label_paths):
    X1, X2, y = [], [], []

    for a, b, l in zip(A_paths, B_paths, label_paths):
        # Read images
        img_A = cv2.imread(a)
        img_B = cv2.imread(b)

        # Resize
        img_A = cv2.resize(img_A, (IMG_SIZE, IMG_SIZE))
        img_B = cv2.resize(img_B, (IMG_SIZE, IMG_SIZE))

        # Normalize
        img_A = img_A / 255.0
        img_B = img_B / 255.0

        # Store separately 👇 THIS IS THE KEY CHANGE
        X1.append(img_A)
        X2.append(img_B)

        # Mask
        mask = cv2.imread(l, 0)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = (mask > 0).astype(np.float32)

        y.append(mask)

    return np.array(X1), np.array(X2), np.array(y)

In [ ]:
X1_train, X2_train, y_train = load_data(train_A, train_B, train_label)
X1_val,   X2_val,   y_val   = load_data(val_A, val_B, val_label)
X1_test,  X2_test,  y_test  = load_data(test_A, test_B, test_label)

print(X1_train.shape, X2_train.shape, y_train.shape)
y_train = np.expand_dims(y_train, axis=-1)
y_val   = np.expand_dims(y_val, axis=-1)
y_test  = np.expand_dims(y_test, axis=-1)

input_shape = (IMG_SIZE, IMG_SIZE, 3)

(445, 128, 128, 3) (445, 128, 128, 3) (445, 128, 128)


In [ ]:
def build_siamese_unet(input_shape):
    inp1 = tf.keras.layers.Input(shape=input_shape)
    inp2 = tf.keras.layers.Input(shape=input_shape)

    # Pretrained encoder
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # Feature maps
    layer_names = [
        "conv1_relu",          # 64x64
        "conv2_block3_out",    # 32x32
        "conv3_block4_out",    # 16x16
        "conv4_block6_out"     # 8x8
    ]

    encoder = tf.keras.Model(
        inputs=base_model.input,
        outputs=[base_model.get_layer(name).output for name in layer_names]
    )

    f1 = encoder(inp1)
    f2 = encoder(inp2)

    # Feature difference
    diff = [
        tf.keras.layers.Lambda(lambda x: tf.abs(x[0] - x[1]))([a, b])
        for a, b in zip(f1, f2)
    ]

    # Start from deepest feature
    x = diff[-1]   # 8x8

    # Decoder

    # 8 → 16
    x = tf.keras.layers.UpSampling2D()(x)
    x = tf.keras.layers.Concatenate()([x, diff[2]])
    x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(x)

    # 16 → 32
    x = tf.keras.layers.UpSampling2D()(x)
    x = tf.keras.layers.Concatenate()([x, diff[1]])
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)

    # 32 → 64
    x = tf.keras.layers.UpSampling2D()(x)
    x = tf.keras.layers.Concatenate()([x, diff[0]])
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)

    # EXTRA UPSAMPLE (critical fix)
    # 64 → 128
    x = tf.keras.layers.UpSampling2D()(x)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)

    # Output
    out = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x)

    return tf.keras.Model([inp1, inp2], out)

In [ ]:
model = build_siamese_unet((128,128,3))
model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_8        │ [(None, 64, 64,   │  8,589,184 │ input_layer_12[0… │
│ (Functional)        │ 64), (None, 32,   │            │ input_layer_13[0… │
│                     │ 32, 256), (None,  │            │                   │
│                     │ 16, 16, 512),     │            │                   │
│                     │ (None, 8, 8,      │            │                   │
│                     │ 1024)]            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_19 (Lambda)  │ (None, 8, 8,      │          0 │ functional_8[0][… │
│                     │ 1024)             │            │ functional_8[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_16    │ (None, 16, 16,    │          0 │ lambda_19[0][0]   │
│ (UpSampling2D)      │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_18 (Lambda)  │ (None, 16, 16,    │          0 │ functional_8[0][… │
│                     │ 512)              │            │ functional_8[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_12      │ (None, 16, 16,    │          0 │ up_sampling2d_16… │
│ (Concatenate)       │ 1536)             │            │ lambda_18[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 16, 16,    │  3,539,200 │ concatenate_12[0… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_17    │ (None, 32, 32,    │          0 │ conv2d_20[0][0]   │
│ (UpSampling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_17 (Lambda)  │ (None, 32, 32,    │          0 │ functional_8[0][… │
│                     │ 256)              │            │ functional_8[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_13      │ (None, 32, 32,    │          0 │ up_sampling2d_17… │
│ (Concatenate)       │ 512)              │            │ lambda_17[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 32, 32,    │    589,952 │ concatenate_13[0… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_18    │ (None, 64, 64,    │          0 │ conv2d_21[0][0]   │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_16 (Lambda)  │ (None, 64, 64,    │          0 │ functional_8[0][… │
│                     │ 64)               │            │ functional_8[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_14      │ (None, 64, 64,    │          0 │ up_sampling2d_18… │
│ (Concatenate)       │ 192)              │            │ lambda_16[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 12,847,489 (49.01 MB)

 Trainable params: 12,816,897 (48.89 MB)

 Non-trainable params: 30,592 (119.50 KB)

In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


In [ ]:
def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    d = dice_loss(y_true, y_pred)
    return bce + d

In [ ]:
def evaluate_metrics(y_true, y_pred):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    acc = accuracy_score(y_true_flat, y_pred_flat)
    prec = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    rec = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)

    intersection = np.logical_and(y_true_flat, y_pred_flat).sum()
    union = np.logical_or(y_true_flat, y_pred_flat).sum()
    iou = intersection / union if union != 0 else 0

    dice = (2 * intersection) / (np.sum(y_true_flat) + np.sum(y_pred_flat) + 1e-6)

    return acc, prec, rec, f1, iou, dice

In [ ]:
model = build_siamese_unet((128,128,3))

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=combined_loss,
    metrics=[
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

In [ ]:
def augment_pair(img1, img2, mask):

    # Horizontal flip
    if np.random.rand() > 0.5:
        img1 = np.fliplr(img1)
        img2 = np.fliplr(img2)
        mask = np.fliplr(mask)

    # Vertical flip
    if np.random.rand() > 0.5:
        img1 = np.flipud(img1)
        img2 = np.flipud(img2)
        mask = np.flipud(mask)

    # Rotation (0, 90, 180, 270)
    k = np.random.randint(0, 4)
    img1 = np.rot90(img1, k)
    img2 = np.rot90(img2, k)
    mask = np.rot90(mask, k)

    # Brightness (ONLY on images, not mask)
    if np.random.rand() > 0.5:
        factor = 0.8 + 0.4 * np.random.rand()
        img1 = np.clip(img1 * factor, 0, 1)
        img2 = np.clip(img2 * factor, 0, 1)

    return img1, img2, mask

In [ ]:
def data_generator(X1, X2, y, batch_size=4):
    while True:
        idx = np.random.randint(0, len(X1), batch_size)

        batch_X1, batch_X2, batch_y = [], [], []

        for i in idx:
            a, b, m = augment_pair(X1[i], X2[i], y[i])

            batch_X1.append(a)
            batch_X2.append(b)
            batch_y.append(m)

        yield (np.array(batch_X1), np.array(batch_X2)), np.array(batch_y)

In [ ]:
model.fit(
    data_generator(X1_train, X2_train, y_train, batch_size=4),
    steps_per_epoch=len(X1_train)//4,
    validation_data=([X1_val, X2_val], y_val),
    epochs=5
)

Epoch 1/5
111/111 ━━━━━━━━━━━━━━━━━━━━ 522s 4s/step - loss: 1.0940 - precision_2: 0.1870 - recall_2: 0.1584 - val_loss: 1.4621 - val_precision_2: 0.0000e+00 - val_recall_2: 0.0000e+00
Epoch 2/5
111/111 ━━━━━━━━━━━━━━━━━━━━ 439s 4s/step - loss: 0.9151 - precision_2: 0.3214 - recall_2: 0.3672 - val_loss: 1.2631 - val_precision_2: 0.0000e+00 - val_recall_2: 0.0000e+00
Epoch 3/5
111/111 ━━━━━━━━━━━━━━━━━━━━ 456s 4s/step - loss: 0.8279 - precision_2: 0.3866 - recall_2: 0.4414 - val_loss: 1.5592 - val_precision_2: 0.0000e+00 - val_recall_2: 0.0000e+00
Epoch 4/5
111/111 ━━━━━━━━━━━━━━━━━━━━ 457s 4s/step - loss: 0.7860 - precision_2: 0.4324 - recall_2: 0.4899 - val_loss: 1.4368 - val_precision_2: 0.2891 - val_recall_2: 0.0016
Epoch 5/5
111/111 ━━━━━━━━━━━━━━━━━━━━ 441s 4s/step - loss: 0.7251 - precision_2: 0.4814 - recall_2: 0.5165 - val_loss: 0.9419 - val_precision_2: 0.3329 - val_recall_2: 0.2757


In [ ]:
preds = model.predict([X1_test, X2_test])
y_pred = (preds > 0.5).astype(int)

acc, prec, rec, f1, iou, dice = evaluate_metrics(y_test, y_pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print("IoU:", iou)
print("Dice:", dice)

4/4 ━━━━━━━━━━━━━━━━━━━━ 28s 6s/step
Accuracy: 0.9349269866943359
Precision: 0.36800030214523116
Recall: 0.25308663353477984
F1: 0.29991278920638176
IoU: 0.1764102378409043
Dice: 0.2999127892048432


In [ ]:
thresholds = np.arange(0.1, 0.9, 0.1)

results = []

for t in thresholds:
    y_pred = (preds > t).astype(int)
    acc, prec, rec, f1, iou, dice = evaluate_metrics(y_test, y_pred)

    results.append([t, prec, rec, f1, iou, dice])

df = pd.DataFrame(results, columns=["Threshold","Precision","Recall","F1","IoU","Dice"])
print(df)

   Threshold  Precision    Recall        F1       IoU      Dice
0        0.1   0.237757  0.587266  0.338479  0.203716  0.338479
1        0.2   0.281426  0.469125  0.351806  0.213449  0.351806
2        0.3   0.313450  0.384907  0.345523  0.208841  0.345523
3        0.4   0.341656  0.315044  0.327811  0.196037  0.327811
4        0.5   0.368000  0.253087  0.299913  0.176410  0.299913
5        0.6   0.394308  0.195423  0.261329  0.150304  0.261329
6        0.7   0.417857  0.138297  0.207814  0.115956  0.207814
7        0.8   0.442144  0.082841  0.139538  0.075002  0.139538
